In [1]:
import torch
import pandas as pd
import json
from transformers import AutoProcessor, VoxtralForConditionalGeneration, BitsAndBytesConfig
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "mistralai/Voxtral-Mini-3B-2507"

print("Loading Base Model for Zero-Shot Evaluation...")
processor = AutoProcessor.from_pretrained(model_id)
# Load in 4-bit or 8-bit to fit in 12GB VRAM during inference
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)
model = VoxtralForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# Load your unseen test dataset
df_test = pd.read_csv("./data/combined_multimodal_dataset_test.csv")
results = []

print("Running Baseline Inference...")
for index, row in tqdm(df_test.iterrows(), total=len(df_test), desc="Zero-Shot Evaluation"):
    audio_path = f"./data/synthesized_test/{row['audio_file_name']}"

    # Zero-Shot format: Just the audio, no ground truth, no dummy turn
    conversation = [
        {"role": "user", "content": [{"type": "audio", "path": audio_path}]}
    ]

    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True, # Tells model to generate the Assistant response
        return_dict=True,
        return_tensors="pt"
    ).to(device)

    # Generate the response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.2,
            top_p=0.95
        )

    decoded_output = processor.batch_decode(
        outputs[:, inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )[0]

    # Attempt to parse JSON (This will likely fail in the baseline)
    valid_json = False
    try:
        # Crude extraction: look for braces
        json_str = decoded_output[decoded_output.find("{"):decoded_output.rfind("}")+1]
        if json_str: # Ensure string is not empty before parsing
            json.loads(json_str)
            valid_json = True
    except:
        pass

    results.append({
        "Command": row["User_Command"],
        "Base_Model_Output": decoded_output.strip(),
        "Valid_JSON": valid_json
    })

# Save for thesis comparison
df_results = pd.DataFrame(results)
df_results.to_csv("./models/baseline_results.csv", index=False)
success_rate = df_results["Valid_JSON"].mean() * 100
print(f"\nBaseline Zero-Shot JSON Success Rate: {success_rate:.2f}%")

Loading Base Model for Zero-Shot Evaluation...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Running Baseline Inference...


Zero-Shot Evaluation:   0%|          | 0/49 [00:00<?, ?it/s]

/home/vito/miniconda3/envs/pyqwen/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")



Baseline Zero-Shot JSON Success Rate: 0.00%
